# Using tiktoken — Demo

Companion to [Tokenization § Real Tokenizer Implementations](https://tanulsingh.github.io/blog/tokenization#real-tokenizer-implementations).

[tiktoken](https://github.com/openai/tiktoken) is OpenAI's official tokenizer library — written in Rust, used by GPT-3.5/4. **No exercises here.** Run top to bottom and observe how a production byte-level BPE tokenizer behaves.

Install:
```
pip install tiktoken
```

## Setup

In [1]:
import tiktoken

## Get a tokenizer

tiktoken ships with multiple pre-trained tokenizers. The most useful for inspection:
- `cl100k_base` — used by GPT-3.5, GPT-4 (turbo), embeddings models
- `o200k_base` — used by GPT-4o
- `r50k_base` — used by GPT-3 (legacy)

In [2]:
enc = tiktoken.get_encoding('cl100k_base')
print(f"vocab size: {enc.n_vocab}")

vocab size: 100277


## Encode and decode

`enc.encode(text)` returns a list of integer token IDs. `enc.decode([ids])` reverses it.

In [3]:
text = "The cat sat on the mat."
tokens = enc.encode(text)
print("token IDs:", tokens)
print("decoded:  ", enc.decode(tokens))

token IDs: [791, 8415, 7731, 389, 279, 5634, 13]
decoded:   The cat sat on the mat.


## See the actual token strings

Use `enc.decode_single_token_bytes(id)` to see what each integer token represents.

Watch for the leading-space convention: in cl100k_base, ` cat` (with leading space) is a different token from `cat` (without). The byte-level encoding treats the space as part of the token.

In [4]:
for tid in tokens:
    piece = enc.decode_single_token_bytes(tid).decode('utf-8', errors='replace')
    print(f"{tid:>6}  {piece!r}")
# Notice: ' cat' (with leading space) is one token, ' sat' is another.
# Spaces stick to the following word — that's the byte-level convention.

   791  'The'
  8415  ' cat'
  7731  ' sat'
   389  ' on'
   279  ' the'
  5634  ' mat'
    13  '.'


## Multilingual compression

The big practical observation: GPT-4's tokenizer compresses different languages very differently.

In [5]:
sentences = {
    'English':  "I love machine learning",
    'Hindi':    "मुझे मशीन लर्निंग पसंद है",
    'Japanese': "私は機械学習が好きです",
}

print(f"{'Language':<10} {'chars':>6} {'tokens':>8} {'chars/token':>14}")
print("-" * 45)
for lang, text in sentences.items():
    n_chars = len(text)
    n_tokens = len(enc.encode(text))
    print(f"{lang:<10} {n_chars:>6} {n_tokens:>8} {n_chars/n_tokens:>14.2f}")
# Even GPT-4's 100K vocab compresses non-Latin scripts much worse than English.

Language    chars   tokens    chars/token
---------------------------------------------
English        23        4           5.75
Hindi          25       27           0.93
Japanese       11       16           0.69


## Code-specific tokenization

cl100k was trained with a lot of code in the corpus. Common code patterns get nice short encodings.

In [6]:
snippets = [
    'def __init__(self):',
    'import torch',
    '    return None',
    'return only nothing',
]
for s in snippets:
    ids = enc.encode(s)
    pieces = [enc.decode_single_token_bytes(i).decode('utf-8', errors='replace') for i in ids]
    print(f"{s!r:<28} -> {len(ids):>2} tokens: {pieces}")
# '    return None' typically uses fewer tokens than 'return only nothing' —
# common code patterns get short encodings, ordinary prose doesn't.

'def __init__(self):'        ->  6 tokens: ['def', ' __', 'init', '__(', 'self', '):']
'import torch'               ->  2 tokens: ['import', ' torch']
'    return None'            ->  3 tokens: ['   ', ' return', ' None']
'return only nothing'        ->  3 tokens: ['return', ' only', ' nothing']


## Practical use: counting tokens for API cost estimation

If you're calling OpenAI's API, you're billed per token. tiktoken lets you count tokens locally before making the call.

In [7]:
def estimated_cost(text, dollars_per_1k_tokens):
    return len(enc.encode(text)) / 1000 * dollars_per_1k_tokens

sample = ("Tokenization is how language models read text. They don't see characters "
          "or words; they see token IDs. The choice of tokenizer determines how many "
          "tokens a given input becomes — which directly drives both API cost and how "
          "much fits in the context window. ") * 10

n_tokens = len(enc.encode(sample))
print(f"sample: {len(sample)} chars, {n_tokens} tokens")
print(f"cost at $0.01/1k tokens: ${estimated_cost(sample, 0.01):.4f}")
print(f"cost at $0.03/1k tokens: ${estimated_cost(sample, 0.03):.4f}")

sample: 2550 chars, 501 tokens
cost at $0.01/1k tokens: $0.0050
cost at $0.03/1k tokens: $0.0150


## Takeaway

tiktoken is a battle-tested implementation of byte-level BPE. The from-scratch version you built in notebook 02 has the same algorithmic core — what tiktoken adds is:

1. **A pre-trained vocabulary** (cl100k = 100K tokens trained on a massive multilingual + code corpus)
2. **Speed** (Rust implementation, ~10x faster than pure Python)
3. **Special tokens** for chat formatting (`<|im_start|>`, etc.)
4. **A regex-based pre-tokenizer** that splits text into chunks before applying BPE — the exact regex is part of the tokenizer's specification

If you're shipping production code that talks to OpenAI: use tiktoken. If you're learning how it works: notebook 02 has the algorithm itself.